In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.optimize import minimize, lsq_linear
from scipy.special import expit
from sklearn.model_selection import KFold

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import AutoMinorLocator, FormatStrFormatter, MaxNLocator, MultipleLocator
from scipy.spatial import cKDTree

MODEL_NAME = 'SE-Hurdle-S'
MODEL_TAG = 'self_exciting_hurdle_s_vs_hurdle_ar1'

SEED = 63
EPS = 0.49
Y = 20

SELF_RESIDUAL_DIST = 'normal' 
HURDLE_RESIDUAL_DIST = 'normal'
CONSTRAIN_EXCITATION = True
HURDLE_NAME = 'Hurdle-AR(1)-S'

RHO_GRID = np.linspace(0.0, 0.98, 51)
N_SPLITS = 5
N_SIM = 10000
N_RANK_REPS = 200
N_BOOT = 1000
DPI = 300

TASK_ROOT = Path.cwd().parent
PROJ_ROOT = TASK_ROOT.parent
OUTPUT = TASK_ROOT/'output'
RESULT_DIR = OUTPUT / 'results'
FIG_DIR = OUTPUT/'figures'
DATA = TASK_ROOT/'input'

In [ ]:
raw = pd.read_csv(DATA/'df_traj_all.csv')

raw['CareerAgeZero'] = pd.to_numeric(raw['CareerAgeZero'], errors='coerce')
raw['pubs_adj'] = pd.to_numeric(raw['pubs_adj'], errors='coerce').clip(lower=0)
raw = raw.dropna(subset=['dblp_id', 'CareerAgeZero', 'pubs_adj'])

full_ids = (raw.groupby('dblp_id')['CareerAgeZero'].max().loc[lambda s: s.eq(Y)].index)

prepared = raw[raw['dblp_id'].isin(full_ids)].copy()
prepared = (prepared.groupby(['dblp_id', 'CareerAgeZero'], as_index=False)['pubs_adj'].sum())

panel = ( prepared.pivot(index='dblp_id', columns='CareerAgeZero', values='pubs_adj').reindex(columns=np.arange(Y + 1)).sort_index())

missing_person_years = int(panel.isna().sum().sum())
panel = panel.fillna(0.0)

Q_EMP = panel.to_numpy(dtype=float)
EMP_IDS = panel.index.to_numpy()
N_EMP = Q_EMP.shape[0]


In [ ]:
STAGE_ORDER = ['0', '1-4', '5-7', '8-20']
STAGE_TRANSITIONS = {
    '0': np.array([0]),
    '1-4': np.arange(1, 5),
    '5-7': np.arange(5, 8),
    '8-20': np.arange(8, 20)}

def transition_stage(t):
    if t == 0:
        return '0'
    if 1 <= t <= 4:
        return '1-4'
    if 5 <= t <= 7:
        return '5-7'
    if 8 <= t <= 19:
        return '8-20'
    raise ValueError(f'No stage for transition year {t}')

def target_stage(target_year):
    return transition_stage(target_year - 1)

def history_panel(q, rho):
    n, years = q.shape
    history = np.zeros_like(q, dtype=float)
    numerator = np.zeros(n)
    denominator = np.zeros(n)

    for target_year in range(1, years):
        history[:, target_year] = np.divide(numerator,denominator,out=np.zeros_like(numerator),where=denominator > 0)
        current_year = target_year - 1
        numerator = rho * numerator + np.log1p(q[:, current_year])
        denominator = rho * denominator + 1.0

    return history

def build_transition_rows(q, history):
    rows = []

    for target_year in range(1, q.shape[1]):
        transition_year = target_year - 1
        q_prev = q[:, transition_year]
        q_now = q[:, target_year]

        rows.append(pd.DataFrame({
            'scholar': np.arange(q.shape[0]),
            'transition_year': transition_year,'target_year': target_year,
            'stage': transition_stage(transition_year),
            'q_prev': q_prev,'q': q_now,
            'active': (q_now > 0).astype(int),
            'x_prev': np.log1p(q_prev),
            'prev_active': (q_prev > 0).astype(int),
            'restart': (q_prev <= 0).astype(int),
            'history': history[:, target_year]}))

    return pd.concat(rows, ignore_index=True)

In [ ]:
CONTINUOUS = {'x_prev', 'history'}
RIDGE = 1e-6
MIN_PROB = 1e-9

def fit_scaler(data, feature_names):
    means = {}
    scales = {}

    for name in feature_names:
        if name in CONTINUOUS:
            means[name] = float(data[name].mean())
            scale = float(data[name].std(ddof=0))
            scales[name] = scale if np.isfinite(scale) and scale > 1e-8 else 1.0
        else:
            means[name] = 0.0
            scales[name] = 1.0

    return means, scales

def design_matrix(data, feature_names, means, scales):
    columns = [np.ones(len(data))]

    for name in feature_names:
        columns.append((data[name].to_numpy(dtype=float) - means[name]) / scales[name])

    return np.column_stack(columns)

def array_design(values, spec):
    n = len(next(iter(values.values())))
    columns = [np.ones(n)]

    for name in spec['feature_names']:
        columns.append((np.asarray(values[name], dtype=float) - spec['means'][name])/ spec['scales'][name])

    return np.column_stack(columns)

def fit_logistic(X, y, constrained_index=None):
    def objective(weights):
        eta = np.clip(X @ weights, -35, 35)
        p = expit(eta)
        nll = -np.sum(y * np.log(p + 1e-12)+ (1 - y) * np.log(1 - p + 1e-12))
        nll += RIDGE * np.sum(weights[1:] ** 2)

        gradient = X.T @ (p - y)
        gradient[1:] += 2 * RIDGE * weights[1:]
        return nll, gradient

    bounds = [(None, None)] * X.shape[1]
    if constrained_index is not None:
        bounds[constrained_index] = (0, None)

    result = minimize(objective,np.zeros(X.shape[1]),jac=True,method='L-BFGS-B',bounds=bounds)

    if not result.success:
        print(f'Logistic warning: {result.message}')

    return result.x

def fit_stage(stage_data, use_history=True, constrain_history=True):
    activity_features = ['x_prev', 'prev_active']
    positive_features = ['x_prev', 'restart']

    if use_history:
        activity_features.append('history')
        positive_features.append('history')

    act_means, act_scales = fit_scaler(stage_data, activity_features)
    X_act = design_matrix(stage_data, activity_features, act_means, act_scales)
    y_act = stage_data['active'].to_numpy(dtype=float)

    act_history_index = None
    if use_history and constrain_history:
        act_history_index = 1 + activity_features.index('history')

    activity_coef = fit_logistic(X_act,y_act,constrained_index=act_history_index)

    positive = stage_data[stage_data['active'].eq(1)].copy()
    pos_means, pos_scales = fit_scaler(positive, positive_features)
    X_pos = design_matrix(positive, positive_features, pos_means, pos_scales)
    y_pos = np.log(positive['q'].to_numpy(dtype=float))

    lower = np.full(X_pos.shape[1], -np.inf)
    upper = np.full(X_pos.shape[1], np.inf)

    if use_history and constrain_history:
        lower[1 + positive_features.index('history')] = 0.0

    positive_fit = lsq_linear(X_pos,y_pos,bounds=(lower, upper),lsq_solver='exact')

    positive_coef = positive_fit.x
    residuals = y_pos - X_pos @ positive_coef
    residuals = residuals - residuals.mean()
    sigma = max(float(np.sqrt(np.mean(residuals ** 2))), 1e-8)
    laplace_scale = max(float(np.mean(np.abs(residuals))), 1e-8)

    return {'activity': {
            'feature_names': activity_features,
            'means': act_means,
            'scales': act_scales,
            'coef': activity_coef,
            'n': len(stage_data)},
        'positive': {
            'feature_names': positive_features,
            'means': pos_means,
            'scales': pos_scales,
            'coef': positive_coef,
            'residuals': residuals,
            'sigma': sigma,
            'laplace_scale': laplace_scale,
            'n': len(positive)}}

def fit_model(q, history, use_history=True, constrain_history=True):
    rows = build_transition_rows(q, history)

    return {stage: fit_stage(rows[rows['stage'].eq(stage)],use_history=use_history,constrain_history=constrain_history) for stage in STAGE_ORDER}

def score_model(model, q, history):
    rows = build_transition_rows(q, history)
    nll = 0.0
    n = 0

    for stage in STAGE_ORDER:
        stage_data = rows[rows['stage'].eq(stage)]
        fitted = model[stage]

        X_act = design_matrix(stage_data,fitted['activity']['feature_names'],fitted['activity']['means'],fitted['activity']['scales'])
        p = np.clip(expit(X_act @ fitted['activity']['coef']),MIN_PROB,1 - MIN_PROB)
        y = stage_data['active'].to_numpy(dtype=float)
        nll -= np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        n += len(stage_data)

        positive = stage_data[stage_data['active'].eq(1)]
        X_pos = design_matrix(positive,fitted['positive']['feature_names'],fitted['positive']['means'],fitted['positive']['scales'])
        y_pos = np.log(positive['q'].to_numpy(dtype=float))
        sigma = fitted['positive']['sigma']
        standardized = (y_pos - X_pos @ fitted['positive']['coef']) / sigma

        nll += np.sum(0.5 * standardized ** 2+ np.log(sigma) + 0.5 * np.log(2 * np.pi))

    return nll / n

def original_scale_coefficients(spec):
    names = spec['feature_names']
    standardized = spec['coef']
    original = {'intercept': float(standardized[0])}

    for j, name in enumerate(names, start=1):
        original[name] = float(standardized[j] / spec['scales'][name])
        original['intercept'] -= (standardized[j]* spec['means'][name]/ spec['scales'][name])

    return original


In [ ]:
folds = list(KFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED).split(Q_EMP))

hurdle_scores = []


profile_rows = []

for rho in RHO_GRID:
    history = history_panel(Q_EMP, rho)
    scores = []

    for train_idx, test_idx in folds:
        fold_model = fit_model(Q_EMP[train_idx],history[train_idx],use_history=True,constrain_history=CONSTRAIN_EXCITATION)
        scores.append(score_model(fold_model,Q_EMP[test_idx],history[test_idx]))

    profile_rows.append({'rho': rho,'mean_nll': np.mean(scores),'se_nll': np.std(scores, ddof=1) / np.sqrt(len(scores))})

rho_profile = pd.DataFrame(profile_rows)
RHO_HAT = float(rho_profile.loc[rho_profile['mean_nll'].idxmin(), 'rho'])
HURDLE_CV_NLL = float(np.mean(hurdle_scores))
SELF_EXCITING_CV_NLL = float(rho_profile['mean_nll'].min())

if 0 < RHO_HAT < 1:
    HALF_LIFE = float(np.log(0.5) / np.log(RHO_HAT))
elif RHO_HAT == 0:
    HALF_LIFE = 0.0
else:
    HALF_LIFE = np.inf

rho_profile.to_csv(RESULT_DIR / 'rho_profile.csv', index=False)

print(f'fit rho: {RHO_HAT:.3f}')
print(f'mem halflife: {HALF_LIFE:.2f} years')
print(f'Self-exciting cv nll: {SELF_EXCITING_CV_NLL:.4f}')
print(f'cv improvement: {HURDLE_CV_NLL - SELF_EXCITING_CV_NLL:.4f}')

In [ ]:
H_EMP = history_panel(Q_EMP, RHO_HAT)

unrestricted_model = fit_model(Q_EMP,H_EMP,use_history=True,constrain_history=False)
self_exciting_model = fit_model(Q_EMP,H_EMP,use_history=True,constrain_history=True)

def parameter_table(model, model_label):
    rows = []

    for stage in STAGE_ORDER:
        for equation in ['activity', 'positive']:
            spec = model[stage][equation]
            original = original_scale_coefficients(spec)
            row = {'model': model_label,'stage': stage,'equation': equation,'n': spec['n'],**original}

            if equation == 'positive':
                row['sigma'] = spec['sigma']
                row['laplace_scale'] = spec['laplace_scale']
            rows.append(row)

    return pd.DataFrame(rows)

def hurdle_parameter_table(model):
    rows = []

    for stage in STAGE_ORDER:
        trans = model[stage]['transition_prob']
        pos = model[stage]['positive']
        rows.append({
            'model': HURDLE_NAME,'stage': stage,
            'p_0_to_0': trans[0, 0],'p_0_to_1': trans[0, 1],'p_1_to_0': trans[1, 0],'p_1_to_1': trans[1, 1],
            'intercept': pos['intercept'],'beta': pos['beta'],
            'sigma': pos['sigma'],
            'laplace_scale': pos['laplace_scale'],
            'positive_n': pos['n'],
            'restart_scale': model[stage]['restart_scale'],
            'restart_n': model[stage]['restart_n']})

    return pd.DataFrame(rows)

params = pd.concat([parameter_table(unrestricted_model, 'Unrestricted history'),parameter_table(self_exciting_model, MODEL_NAME)], ignore_index=True)

params['rho'] = RHO_HAT
params['half_life'] = HALF_LIFE
params.to_csv(RESULT_DIR / 'self_exciting_stage_parameters.csv', index=False)

# display(hurdle_params)
# params

In [ ]:
rng = np.random.default_rng(SEED)
bootstrap_rows = []

for b in range(N_BOOT):
    sampled = rng.integers(0, N_EMP, size=N_EMP)
    q_boot = Q_EMP[sampled]
    h_boot = history_panel(q_boot, RHO_HAT)
    fitted = fit_model(q_boot,h_boot,use_history=True,constrain_history=False)

    table = parameter_table(fitted, 'bootstrap')
    table['bootstrap'] = b
    bootstrap_rows.append(table)

bootstrap_params = pd.concat(bootstrap_rows, ignore_index=True)
bootstrap_params.to_csv(RESULT_DIR / 'history_coefficient_bootstrap.csv',index=False)

bootstrap_ci = (bootstrap_params.groupby(['stage', 'equation'])['history'].quantile([0.025, 0.5, 0.975]).unstack().reset_index().rename(columns={0.025: 'low', 0.5: 'median', 0.975: 'high'}))

In [ ]:
def draw_residuals(spec, size, rng, distribution):
    if distribution == 'empirical':
        return rng.choice(spec['residuals'], size=size, replace=True)
    if distribution == 'normal':
        return rng.normal(0, spec['sigma'], size=size)
    if distribution == 'laplace':
        return rng.laplace(0, spec['laplace_scale'], size=size)
    raise ValueError(f'Unknown residual distribution: {distribution}')

def positive_exponential(scale, size, rng):
    draws = rng.exponential(scale=scale, size=size)
    zero = draws <= 0
    while zero.any():
        draws[zero] = rng.exponential(scale=scale, size=zero.sum())
        zero = draws <= 0
    return draws

def simulate_self_exciting(model, rho, n_sim, seed):
    rng = np.random.default_rng(seed)
    simulated = np.zeros((n_sim, Y + 1), dtype=float)
    simulated[:, 0] = rng.choice(Q_EMP[:, 0], size=n_sim, replace=True)

    numerator = np.zeros(n_sim)
    denominator = np.zeros(n_sim)

    for target_year in range(1, Y + 1):
        history = np.divide(numerator,denominator,out=np.zeros_like(numerator),where=denominator > 0)

        q_prev = simulated[:, target_year - 1]
        values = {'x_prev': np.log1p(q_prev),'prev_active': (q_prev > 0).astype(float),'restart': (q_prev <= 0).astype(float),'history': history}

        stage = target_stage(target_year)
        activity = model[stage]['activity']
        X_act = array_design(values, activity)
        p_active = expit(np.clip(X_act @ activity['coef'], -35, 35))
        active = rng.random(n_sim) < p_active

        if active.any():
            positive = model[stage]['positive']
            active_values = {name: values[name][active] for name in values}
            X_pos = array_design(active_values, positive)
            mean_log_q = X_pos @ positive['coef']
            noise = draw_residuals(positive,active.sum(),rng,SELF_RESIDUAL_DIST)
            log_q = np.clip(mean_log_q + noise, -30, 30)
            simulated[active, target_year] = np.exp(log_q)

        numerator = rho * numerator + np.log1p(q_prev)
        denominator = rho * denominator + 1.0

    return simulated

Q_SELF = simulate_self_exciting(self_exciting_model,RHO_HAT,N_SIM,SEED + 2)

np.save(RESULT_DIR / f'{MODEL_TAG}_trajs.npy', Q_SELF)

In [ ]:
def year_stats(q, label):
    raw = q
    logged = np.log(raw + EPS)

    return pd.DataFrame({
        'model': label,
        'year': np.arange(q.shape[1]),
        'mean': raw.mean(axis=0),
        'median': np.median(raw, axis=0),
        'variance': raw.var(axis=0),
        'zero_fraction': (raw == 0).mean(axis=0),
        'log_mean': logged.mean(axis=0),
        'log_median': np.median(logged, axis=0),
        'log_variance': logged.var(axis=0)})

moment_stats = pd.concat([year_stats(Q_EMP, 'Empirical'),year_stats(Q_SELF, MODEL_NAME)], ignore_index=True)

moment_stats.to_csv(RESULT_DIR / 'year_stats.csv', index=False)